# ElectPyNasa — Massive Image Downloader & Tiling Pipeline for Google Colab

A production-grade pipeline for downloading **multi-gigabyte astronomical images**
(JWST / HST FITS, BigTIFF) from the MAST archive and slicing them into
analytical tiles **without ever loading the full image into RAM**.

---

### Why this notebook exists

Raw observatory products routinely exceed **2–10 GB per file**. Colab's free
runtime has limited RAM (~12 GB) and ephemeral storage. Naive
`requests.get(...).content` downloads and `np.array(image)` loads will crash
the kernel within seconds.

This notebook solves three problems at once:

| Problem | Solution |
|---------|----------|
| **Download fails halfway through a 5 GB file** | HTTP Range resume + chunk-level + session-level retries |
| **No visibility into download health** | Real-time progress bar with rolling speed (MB/s) and ETA |
| **File might be truncated / corrupted** | Size check + SHA-256 + actual readability probe |
| **Tiling a 5 GB image OOMs the kernel** | Memory-mapped reads via `astropy.io.fits` and `tifffile` |
| **MAST URLs are fragile** | `astroquery.mast` search + direct-URL fallback with resume |

---

### Architecture

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        Colab runtime                                    │
│                                                                         │
│  Configuration ─▶ Pipeline orchestrator                                │
│                         │                                               │
│              ┌──────────┼───────────────┬───────────────┐               │
│              ▼          ▼               ▼               ▼               │
│         MastClient  ResumableDownloader  Verifier   MemoryMappedTiler   │
│         (search +   (Range + retries +  (size +     (FITS / TIFF,       │
│          URL extract) speed + ETA)       SHA-256 +   multi-channel,     │
│                                            readable)  manifest CSV)     │
└─────────────────────────────────────────────────────────────────────────┘
                              │
                              ▼
              /content/space_data/  (or /content/drive/MyDrive/...)
                  ├── downloads/   ← verified source files
                  └── tiles/       ← tile_XXXX_yYYY_xXXXX.tif + manifest.csv
```

## Step 1 — Install Dependencies

The cell below installs the scientific imaging stack used by the pipeline:

- `astropy` + `astroquery` — FITS I/O and the official MAST archive client
- `tifffile` — memory-mapped BigTIFF reading
- `tqdm` — progress bars (Colab-aware)
- `requests` — HTTP client with Range support

In [ ]:
# Install scientific dependencies (idempotent — safe to re-run).
import sys, subprocess

def _pip_install(*packages: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

_pip_install(
    "astropy>=5.3",
    "astroquery>=0.4.7",
    "tifffile>=2023.3.15",
    "tqdm>=4.65",
    "requests>=2.31",
    "numpy>=1.23",
)
print("[setup] All dependencies installed.")

## Step 2 — Imports & Global Configuration

All tunable knobs live in the `Config` dataclass below. Edit the values in
**Step 7** (the run cells) — the defaults here are sane for Colab.

In [ ]:
from __future__ import annotations

import os
import sys
import time
import json
import shutil
import hashlib
import logging
import dataclasses
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional, Iterable, Any, Iterator

import numpy as np
import requests
from tqdm.auto import tqdm

print(f"[imports] Python {sys.version.split()[0]} on {sys.platform}")
print(f"[imports] numpy={np.__version__}, requests={requests.__version__}")

## Step 3 — Mount Google Drive (Optional but Recommended)

Mounting Drive gives you **persistent storage** — when the Colab runtime
recycles (which it does every ~12 h on the free tier), anything in
`/content/` is lost. Anything under `/content/drive/MyDrive/` survives.

If you skip this cell, the pipeline will use `/content/space_data/` instead.

In [ ]:
DRIVE_MOUNTED = False
DRIVE_ROOT = Path("/content/drive/MyDrive")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_MOUNTED = DRIVE_ROOT.exists()
    if DRIVE_MOUNTED:
        print(f"[drive] Mounted. Persistent root: {DRIVE_ROOT}")
    else:
        print("[drive] Mount command ran but root not visible — using /content.")
except ImportError:
    print("[drive] Not running on Colab — using /content.")
except Exception as exc:
    print(f"[drive] Mount failed: {exc}. Using /content.")

# Default workspace — override in the run cell if you want Drive.
DEFAULT_WORKSPACE = Path("/content/space_data")
if DRIVE_MOUNTED:
    DEFAULT_WORKSPACE = DRIVE_ROOT / "ElectPyNasa"

DEFAULT_WORKSPACE.mkdir(parents=True, exist_ok=True)
print(f"[drive] Workspace: {DEFAULT_WORKSPACE}")

## Step 4 — Structured Logger

A thin wrapper around the stdlib `logging` module that emits timestamped,
colored lines to the notebook stdout. Every module reports through this
logger so the output reads like a coherent trace, not five `print` styles
fighting each other.

In [ ]:
class PipelineLogger:
    """Colab-friendly structured logger."""

    _INSTANCE: Optional["PipelineLogger"] = None

    def __init__(self, level: int = logging.INFO) -> None:
        self._logger = logging.getLogger("electpynasa.colab")
        self._logger.setLevel(level)
        if not self._logger.handlers:
            handler = logging.StreamHandler(sys.stdout)
            handler.setFormatter(logging.Formatter(
                "%(asctime)s | %(levelname)-7s | %(message)s",
                datefmt="%H:%M:%S",
            ))
            self._logger.addHandler(handler)
            self._logger.propagate = False

    @classmethod
    def get(cls) -> "PipelineLogger":
        if cls._INSTANCE is None:
            cls._INSTANCE = cls()
        return cls._INSTANCE

    def info(self, msg: str, **extra: Any) -> None:
        self._logger.info(self._format(msg, extra))

    def warning(self, msg: str, **extra: Any) -> None:
        self._logger.warning(self._format(msg, extra))

    def error(self, msg: str, **extra: Any) -> None:
        self._logger.error(self._format(msg, extra))

    def success(self, msg: str, **extra: Any) -> None:
        # Green-tinted success line
        self._logger.info("\033[92m" + self._format(msg, extra) + "\033[0m")

    @staticmethod
    def _format(msg: str, extra: dict[str, Any]) -> str:
        if not extra:
            return msg
        kv = " ".join(f"{k}={v}" for k, v in extra.items())
        return f"{msg}  [{kv}]"


log = PipelineLogger.get()
log.info("Logger ready.")

## Step 5 — Resumable Downloader

The heart of the pipeline. Key design points:

- **HTTP Range resume** — if a partial file exists on disk and the server
  supports `Accept-Ranges: bytes`, the download continues from the byte
  offset rather than restarting.
- **Two-tier retry** — *chunk-level* retry (fast, no backoff) handles brief
  socket hiccups without dropping the session; *session-level* retry
  (exponential backoff) handles full connection drops.
- **Rolling speed + ETA** — a custom tqdm callback maintains a 10-second
  sliding window of bytes received and reports `MB/s` and `ETA` in real
  time, not just a percentage.
- **Atomic rename** — the file is written to `<dest>.part` and only renamed
  to `<dest>` after the size check passes, so a partial file is never
  mistaken for a complete one.

In [ ]:
class RollingSpeed:
    """Sliding-window byte-rate estimator for real-time speed/ETA display."""

    def __init__(self, window_seconds: float = 10.0) -> None:
        self._window = window_seconds
        self._samples: list[tuple[float, int]] = []  # (timestamp, cumulative_bytes)

    def update(self, cumulative_bytes: int, now: Optional[float] = None) -> None:
        t = now if now is not None else time.monotonic()
        self._samples.append((t, cumulative_bytes))
        cutoff = t - self._window
        while self._samples and self._samples[0][0] < cutoff:
            self._samples.pop(0)

    def rate(self) -> float:
        """Return current bytes/sec, or 0 if insufficient samples."""
        if len(self._samples) < 2:
            return 0.0
        t0, b0 = self._samples[0]
        t1, b1 = self._samples[-1]
        dt = t1 - t0
        if dt <= 0:
            return 0.0
        return (b1 - b0) / dt


class ResumableDownloader:
    """
    HTTP downloader with Range resume, two-tier retries, and real-time
    speed/ETA reporting. Optimized for multi-gigabyte scientific files.
    """

    def __init__(
        self,
        url: str,
        dest_path: str | Path,
        *,
        chunk_size: int = 8 * 1024 * 1024,        # 8 MiB — optimal for big files
        max_session_retries: int = 8,
        max_chunk_retries: int = 5,
        backoff_factor: float = 2.0,
        backoff_max: float = 60.0,
        timeout: float = 30.0,
        headers: Optional[dict[str, str]] = None,
    ) -> None:
        self.url = url
        self.dest_path = Path(dest_path)
        self.chunk_size = chunk_size
        self.max_session_retries = max_session_retries
        self.max_chunk_retries = max_chunk_retries
        self.backoff_factor = backoff_factor
        self.backoff_max = backoff_max
        self.timeout = timeout
        self.extra_headers = headers or {}

        # Populated after a HEAD probe; useful for verification later.
        self.remote_size: int = 0
        self.accepts_ranges: bool = False
        self.etag: Optional[str] = None

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------
    def download(self) -> Path:
        """Download the file with full resume + retry. Returns the final path."""
        log.info(f"Starting download: {self.url}")
        log.info(f"Destination: {self.dest_path}")

        self.dest_path.parent.mkdir(parents=True, exist_ok=True)
        self._probe_remote()

        # Already complete?
        if self.remote_size > 0 and self.dest_path.exists() and self.dest_path.stat().st_size == self.remote_size:
            log.success(f"File already complete on disk ({self._fmt_size(self.remote_size)}).")
            return self.dest_path

        # Resume from partial?
        part_path = self.dest_path.with_suffix(self.dest_path.suffix + ".part")
        offset = part_path.stat().st_size if part_path.exists() else 0
        if offset > 0 and self.accepts_ranges:
            log.info(f"Resuming from {self._fmt_size(offset)} / {self._fmt_size(self.remote_size)} "
                     f"({100 * offset / max(self.remote_size, 1):.1f}%)")
        elif offset > 0 and not self.accepts_ranges:
            log.warning("Server doesn't support Range; discarding partial file.")
            part_path.unlink(missing_ok=True)
            offset = 0

        # Session-level retry loop
        for attempt in range(1, self.max_session_retries + 1):
            try:
                self._download_session(part_path, offset)
                # Success — verify size, then atomic rename
                self._verify_part_size(part_path)
                part_path.replace(self.dest_path)
                log.success(f"Download complete: {self.dest_path} "
                            f"({self._fmt_size(self.dest_path.stat().st_size)})")
                return self.dest_path
            except KeyboardInterrupt:
                raise
            except Exception as exc:
                if part_path.exists():
                    offset = part_path.stat().st_size
                delay = min(self.backoff_factor ** attempt, self.backoff_max)
                log.warning(f"Session {attempt}/{self.max_session_retries} failed: {exc}")
                log.warning(f"Partial progress: {self._fmt_size(offset)}. Retrying in {delay:.1f}s...")
                if attempt == self.max_session_retries:
                    raise
                time.sleep(delay)

        raise RuntimeError("Unreachable: retry loop exhausted without return or raise.")

    # ------------------------------------------------------------------
    # Internals
    # ------------------------------------------------------------------
    def _probe_remote(self) -> None:
        """Discover total size, Range support, and ETag via HEAD (fallback to GET)."""
        try:
            r = requests.head(self.url, allow_redirects=True, timeout=self.timeout,
                              headers=self.extra_headers)
            if r.status_code >= 400:
                r.raise_for_status()
            headers = r.headers
        except Exception:
            r = requests.get(self.url, stream=True, allow_redirects=True,
                             timeout=self.timeout, headers=self.extra_headers)
            r.raise_for_status()
            headers = r.headers
            r.close()

        self.remote_size = int(headers.get("content-length", 0))
        self.accepts_ranges = "bytes" in headers.get("accept-ranges", "").lower()
        self.etag = headers.get("etag")
        log.info(f"Remote size: {self._fmt_size(self.remote_size)} | "
                 f"Range support: {self.accepts_ranges} | ETag: {self.etag or 'n/a'}")
        if self.remote_size == 0:
            log.warning("Content-Length missing — resume will be limited.")

    def _download_session(self, part_path: Path, offset: int) -> None:
        """Single download session with chunk-level retries."""
        headers = dict(self.extra_headers)
        if offset > 0 and self.accepts_ranges:
            headers["Range"] = f"bytes={offset}-"

        with requests.get(self.url, stream=True, timeout=self.timeout, headers=headers) as r:
            if offset > 0 and self.accepts_ranges:
                if r.status_code != 206:
                    log.warning(f"Server returned {r.status_code} instead of 206; restarting from 0.")
                    offset = 0
                    r.close()
                    return self._download_session(part_path, 0)
            else:
                r.raise_for_status()

            mode = "ab" if (offset > 0 and self.accepts_ranges) else "wb"
            speed = RollingSpeed()
            pbar = tqdm(
                total=self.remote_size or None,
                initial=offset,
                unit="B", unit_scale=True, unit_divisor=1024,
                desc=self.dest_path.name,
            )
            written = offset
            try:
                with open(part_path, mode) as f:
                    chunk_retries = 0
                    for chunk in r.iter_content(chunk_size=self.chunk_size):
                        if not chunk:
                            continue
                        try:
                            f.write(chunk)
                            f.flush()
                        except IOError as exc:
                            chunk_retries += 1
                            if chunk_retries > self.max_chunk_retries:
                                raise
                            log.warning(f"Chunk write failed (attempt {chunk_retries}): {exc}")
                            time.sleep(1.0)
                            continue
                        written += len(chunk)
                        pbar.update(len(chunk))
                        speed.update(written)
                        if written % (64 * self.chunk_size) == 0:
                            rate = speed.rate()
                            if rate > 0 and self.remote_size > 0:
                                eta_s = (self.remote_size - written) / rate
                                pbar.set_postfix({
                                    "speed": f"{rate / 1024 / 1024:.1f} MB/s",
                                    "ETA": self._fmt_duration(eta_s),
                                })
                            else:
                                pbar.set_postfix({"speed": f"{rate / 1024 / 1024:.1f} MB/s"})
            finally:
                # Final postfix update
                rate = speed.rate()
                pbar.set_postfix({"speed": f"{rate / 1024 / 1024:.1f} MB/s"})
                pbar.close()

    def _verify_part_size(self, part_path: Path) -> None:
        """Verify the .part file has the expected size before atomic rename."""
        if self.remote_size <= 0:
            return  # Can't verify without a known remote size.
        actual = part_path.stat().st_size
        if actual != self.remote_size:
            raise IOError(f"Size mismatch after download: expected {self.remote_size}, got {actual}.")

    @staticmethod
    def _fmt_size(num_bytes: int) -> str:
        for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
            if abs(num_bytes) < 1024.0:
                return f"{num_bytes:.1f} {unit}"
            num_bytes /= 1024.0
        return f"{num_bytes:.1f} PiB"

    @staticmethod
    def _fmt_duration(seconds: float) -> str:
        if seconds < 0:
            return "?"
        s = int(seconds)
        if s < 60:
            return f"{s}s"
        if s < 3600:
            return f"{s // 60}m{s % 60}s"
        return f"{s // 3600}h{(s % 3600) // 60}m"

log.info("ResumableDownloader class loaded.")

## Step 6 — File Verifier

Three layers of integrity checking, from cheapest to most expensive:

1. **Size check** — does the on-disk byte count match `Content-Length`?
2. **SHA-256 hash** — streamed 1 MiB at a time, never loads the file into RAM.
3. **Readability probe** — actually open the file with `astropy` / `tifffile`
   and read a small slice. A truncated FITS or TIFF can pass the size check
   but still be unreadable.

In [ ]:
class FileVerifier:
    """Three-layer integrity verifier: size + SHA-256 + readability."""

    CHUNK = 1024 * 1024  # 1 MiB

    @classmethod
    def verify(cls, path: str | Path, *, expected_size: int = 0,
               expected_sha256: Optional[str] = None) -> bool:
        path = Path(path)
        if not path.exists():
            log.error(f"File not found: {path}")
            return False

        # Layer 1: size
        if expected_size > 0:
            actual = path.stat().st_size
            if actual != expected_size:
                log.error(f"Size mismatch: expected {expected_size}, got {actual}.")
                return False
            log.info(f"Size OK: {actual} bytes.")

        # Layer 2: SHA-256
        if expected_sha256:
            actual_hash = cls.sha256(path)
            if actual_hash != expected_sha256.lower():
                log.error(f"SHA-256 mismatch: expected {expected_sha256}, got {actual_hash}.")
                return False
            log.info(f"SHA-256 OK: {actual_hash}")
        else:
            log.info("No expected SHA-256 provided — skipping hash check.")

        # Layer 3: readability
        if not cls.is_readable(path):
            log.error(f"File exists but cannot be read as FITS/TIFF: {path}")
            return False
        log.success(f"Integrity verified: {path.name}")
        return True

    @classmethod
    def sha256(cls, path: str | Path) -> str:
        """Streamed SHA-256 (constant memory, ~500 MB/s on Colab)."""
        h = hashlib.sha256()
        path = Path(path)
        with tqdm(total=path.stat().st_size, unit="B", unit_scale=True,
                  desc=f"SHA-256 {path.name}") as pbar:
            with open(path, "rb") as f:
                while True:
                    block = f.read(cls.CHUNK)
                    if not block:
                        break
                    h.update(block)
                    pbar.update(len(block))
        return h.hexdigest()

    @classmethod
    def is_readable(cls, path: str | Path) -> bool:
        """Open the file with the appropriate library and read a small slice."""
        path = Path(path)
        ext = path.suffix.lower()
        try:
            if ext in (".fits", ".fit"):
                from astropy.io import fits
                with fits.open(path, memmap=True) as hdul:
                    for hdu in hdul:
                        if isinstance(hdu.data, np.ndarray) and hdu.data is not None:
                            _ = hdu.data[0:1, 0:1]  # force a tiny read
                            return True
                return False
            if ext in (".tif", ".tiff"):
                import tifffile
                with tifffile.TiffFile(path) as tif:
                    arr = tif.asarray(out="memmap")
                    _ = arr[tuple(slice(0, 1) for _ in arr.shape)]
                    return True
            log.warning(f"Unknown extension {ext!r} — skipping readability check.")
            return True
        except Exception as exc:
            log.error(f"Readability check failed: {exc}")
            return False

log.info("FileVerifier class loaded.")

## Step 7 — MAST Archive Client

A thin wrapper around `astroquery.mast.Observations` that:

- Searches MAST by target name + instrument + filter (or by observation ID).
- Filters down to **Level 3 calibrated mosaics** (`i2d.fits`) — these are the
  science-ready products you actually want, not the raw `uncal` files.
- **Extracts the direct download URL** for each product so the
  `ResumableDownloader` can fetch it with full resume + retry support.
- Optionally attaches a MAST API token (required for some proprietary JWST
  data — get one at https://auth.mast.stsci.edu/).

In [ ]:
class MastClient:
    """Wrapper around astroquery.mast that yields direct, resumable URLs."""

    MAST_DOWNLOAD_BASE = "https://mast.stsci.edu/api/v0.1/Download/file?uri="

    def __init__(self, api_token: Optional[str] = None) -> None:
        self._token = api_token
        self._configured = False

    def _ensure_configured(self) -> None:
        if self._configured:
            return
        from astroquery.mast import Observations
        if self._token:
            Observations.login(token=self._token)
            log.info("MAST login configured with API token.")
        else:
            log.info("MAST anonymous access (public data only).")
        self._configured = True

    def search_by_target(self, target_name: str, *, instrument: str = "NIRCAM",
                         filter_name: Optional[str] = None,
                         obs_collection: str = "JWST",
                         limit: int = 10) -> list[dict]:
        """Search MAST by target + instrument + filter. Returns a list of dicts."""
        self._ensure_configured()
        from astroquery.mast import Observations

        criteria = {
            "target_name": target_name,
            "obs_collection": obs_collection,
            "instrument_name": instrument,
        }
        if filter_name:
            criteria["filters"] = filter_name

        log.info(f"Querying MAST: {criteria}")
        obs_table = Observations.query_criteria(**criteria)
        log.info(f"Found {len(obs_table)} observation(s).")
        return [dict(zip(obs_table.colnames, row)) for row in obs_table[:limit]]

    def list_i2d_products(self, observation: dict) -> list[dict]:
        """For a single observation row, return its Level-3 i2d.fits products."""
        self._ensure_configured()
        from astroquery.mast import Observations

        # Reconstruct a minimal Table row to satisfy astroquery's API.
        obs_id = observation.get("obsid") or observation.get("obsID")
        if obs_id is None:
            raise ValueError("Observation dict must contain 'obsid' or 'obsID'.")

        products = Observations.get_product_list(obs_id)
        mask = (
            (products["calib_level"] == 3)
            & (products["productSubGroupDescription"] == "I2D")
            & (products["extension"] == "fits")
        )
        filtered = products[mask]
        log.info(f"Observation {obs_id}: {len(filtered)} i2d.fits product(s).")
        return [dict(zip(filtered.colnames, row)) for row in filtered]

    def direct_url(self, product: dict) -> str:
        """Build the direct, resumable download URL for a MAST product."""
        uri = product.get("uri") or product.get("productFilename")
        if not uri:
            raise ValueError("Product dict has no 'uri' or 'productFilename'.")
        # MAST URIs look like 'mast:JWST/product/jw02731_..._i2d.fits'
        if uri.startswith("mast:"):
            return self.MAST_DOWNLOAD_BASE + uri
        # If it's already a full URL, return as-is.
        if uri.startswith("http"):
            return uri
        # Otherwise, treat as a bare filename under the JWST product path.
        return self.MAST_DOWNLOAD_BASE + f"mast:JWST/product/{uri}"

    def download_via_astroquery(self, products: list[dict],
                                download_dir: str | Path) -> list[Path]:
        """Fallback path: let astroquery handle the download (no resume)."""
        self._ensure_configured()
        from astroquery.mast import Observations
        from astropy.table import Table

        log.warning("Using astroquery fallback download (no resume support).")
        tbl = Table(products)
        manifest = Observations.download_products(tbl, download_dir=str(download_dir))
        return [Path(row["Local Path"]) for row in manifest]

log.info("MastClient class loaded.")

## Step 8 — Memory-Mapped Tiler

Slices a giant FITS or TIFF into smaller tiles **without ever loading the
full image into RAM**. Both `astropy.io.fits` (with `memmap=True`) and
`tifffile.TiffFile.asarray(out='memmap')` return numpy arrays backed by the
file — slicing them only reads the relevant bytes from disk.

Features:

- **Multi-channel aware** — handles `(H, W)`, `(H, W, 3)`, and `(C, H, W)` shapes.
- **Preserves dtype** — tiles keep the original bit depth (no surprise
  upcasts to `float32`).
- **Tile manifest CSV** — every tile is recorded with its `(y, x)` origin,
  shape, and source file, so downstream code can stitch them back together.
- **Skip-empty tiles** — tiles whose sum is zero (pure background) are
  skipped to save disk space.
- **Resume support** — if a tile file already exists and matches the
  expected size, it is skipped on re-runs.

In [ ]:
import csv

class MemoryMappedTiler:
    """Memory-mapped FITS/TIFF tiler with manifest generation and resume."""

    def __init__(self, tile_size: int = 1024, overlap: int = 0,
                 skip_empty: bool = False, skip_existing: bool = True,
                 empty_threshold: float = 0.0) -> None:
        if tile_size <= 0:
            raise ValueError("tile_size must be > 0")
        if not 0 <= overlap < tile_size:
            raise ValueError("overlap must be in [0, tile_size)")
        self.tile_size = tile_size
        self.overlap = overlap
        self.skip_empty = skip_empty
        self.skip_existing = skip_existing
        self.empty_threshold = empty_threshold

    # ------------------------------------------------------------------
    # Public entry points
    # ------------------------------------------------------------------
    def tile(self, source_path: str | Path, out_dir: str | Path) -> Path:
        """Dispatch to FITS or TIFF tiler. Returns the manifest CSV path."""
        source_path = Path(source_path)
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        ext = source_path.suffix.lower()
        if ext in (".fits", ".fit"):
            return self._tile_fits(source_path, out_dir)
        if ext in (".tif", ".tiff"):
            return self._tile_tiff(source_path, out_dir)
        raise ValueError(f"Unsupported tiling format: {ext}")

    # ------------------------------------------------------------------
    # FITS
    # ------------------------------------------------------------------
    def _tile_fits(self, source_path: Path, out_dir: Path) -> Path:
        from astropy.io import fits
        log.info(f"Opening FITS (memmap): {source_path}")
        with fits.open(source_path, memmap=True) as hdul:
            img_data = None
            for hdu in hdul:
                if isinstance(hdu.data, np.ndarray) and hdu.data is not None and hdu.data.ndim >= 2:
                    img_data = hdu.data
                    break
            if img_data is None:
                raise ValueError("No 2D image data found in FITS.")
            return self._slice_and_write(img_data, source_path, out_dir)

    # ------------------------------------------------------------------
    # TIFF
    # ------------------------------------------------------------------
    def _tile_tiff(self, source_path: Path, out_dir: Path) -> Path:
        import tifffile
        log.info(f"Opening TIFF (memmap): {source_path}")
        with tifffile.TiffFile(source_path) as tif:
            img_data = tif.asarray(out="memmap")
            return self._slice_and_write(img_data, source_path, out_dir)

    # ------------------------------------------------------------------
    # Shared slicing logic
    # ------------------------------------------------------------------
    def _slice_and_write(self, img_data: np.ndarray, source_path: Path,
                         out_dir: Path) -> Path:
        # Normalize shape: we always iterate over (H, W); extra leading/trailing
        # axes are preserved per-tile.
        if img_data.ndim == 2:
            h, w = img_data.shape
        elif img_data.ndim == 3:
            # Could be (C, H, W) or (H, W, C). Heuristic: the last two axes
            # are spatial if the first axis is small; otherwise the first two.
            if img_data.shape[0] <= 4 and img_data.shape[1] > 4:
                h, w = img_data.shape[1], img_data.shape[2]
            else:
                h, w = img_data.shape[0], img_data.shape[1]
        else:
            raise ValueError(f"Unsupported image ndim: {img_data.ndim}")

        log.info(f"Image shape: {img_data.shape}  →  H={h}, W={w}")

        step = self.tile_size - self.overlap
        ys = list(range(0, max(h - self.overlap, 1), step))
        xs = list(range(0, max(w - self.overlap, 1), step))
        # Make sure the last tile reaches the edge.
        if ys and ys[-1] + self.tile_size < h:
            ys.append(h - self.tile_size if h >= self.tile_size else 0)
        if xs and xs[-1] + self.tile_size < w:
            xs.append(w - self.tile_size if w >= self.tile_size else 0)
        # Deduplicate (overlap can cause duplicates near the edge).
        ys = sorted(set(ys))
        xs = sorted(set(xs))

        total = len(ys) * len(xs)
        log.info(f"Will produce up to {total} tiles "
                 f"({len(ys)} rows × {len(xs)} cols, tile={self.tile_size}, overlap={self.overlap}).")

        manifest_path = out_dir / "manifest.csv"
        written = 0
        skipped_existing = 0
        skipped_empty = 0

        with open(manifest_path, "w", newline="") as f_csv:
            writer = csv.writer(f_csv)
            writer.writerow([
                "tile_index", "y", "x", "height", "width",
                "dtype", "source", "tile_file",
            ])
            with tqdm(total=total, desc="Tiling", unit="tile") as pbar:
                tile_index = 0
                for y in ys:
                    for x in xs:
                        y_end = min(y + self.tile_size, h)
                        x_end = min(x + self.tile_size, w)
                        # Slice — for ndim==3 the slicing keeps all channels.
                        if img_data.ndim == 2:
                            tile = img_data[y:y_end, x:x_end]
                        elif img_data.ndim == 3:
                            if img_data.shape[0] <= 4 and img_data.shape[1] > 4:
                                tile = img_data[:, y:y_end, x:x_end]
                            else:
                                tile = img_data[y:y_end, x:x_end, :]
                        else:
                            tile = img_data[y:y_end, x:x_end]

                        # Skip empty tiles (optional).
                        if self.skip_empty:
                            try:
                                tile_sum = float(np.asarray(tile).sum())
                            except Exception:
                                tile_sum = 0.0
                            if tile_sum <= self.empty_threshold:
                                skipped_empty += 1
                                pbar.update(1)
                                tile_index += 1
                                continue

                        tile_name = f"tile_{tile_index:05d}_y{y:06d}_x{x:06d}.tif"
                        tile_path = out_dir / tile_name

                        # Resume: skip if file exists and matches expected shape.
                        if self.skip_existing and tile_path.exists():
                            try:
                                import tifffile as _t
                                existing = _t.imread(str(tile_path))
                                if existing.shape == tile.shape:
                                    skipped_existing += 1
                                    writer.writerow([tile_index, y, x,
                                                     tile.shape[0], tile.shape[1],
                                                     str(tile.dtype),
                                                     source_path.name, tile_name])
                                    pbar.update(1)
                                    tile_index += 1
                                    continue
                            except Exception:
                                pass  # Re-write if we can't verify.

                        # Write — preserve original dtype for fidelity.
                        import tifffile as _t
                        _t.imwrite(str(tile_path), np.ascontiguousarray(tile))
                        writer.writerow([tile_index, y, x,
                                         tile.shape[0], tile.shape[1],
                                         str(tile.dtype),
                                         source_path.name, tile_name])
                        written += 1
                        pbar.update(1)
                        tile_index += 1

        log.success(
            f"Tiling complete: wrote={written}, skipped_existing={skipped_existing}, "
            f"skipped_empty={skipped_empty}, manifest={manifest_path}"
        )
        return manifest_path

log.info("MemoryMappedTiler class loaded.")

## Step 9 — Pipeline Orchestrator

Ties the modules together. Two convenience methods:

- `Pipeline.run_url(url, name)` — download a single direct URL, verify, tile.
- `Pipeline.run_mast(target, instrument, filter)` — search MAST, download
  every matching `i2d.fits`, verify, tile each one.

In [ ]:
@dataclass
class PipelineConfig:
    workspace: Path = DEFAULT_WORKSPACE
    downloads_subdir: str = "downloads"
    tiles_subdir: str = "tiles"
    tile_size: int = 1024
    tile_overlap: int = 64
    skip_empty_tiles: bool = False
    mast_api_token: Optional[str] = None
    # Downloader
    chunk_size: int = 8 * 1024 * 1024
    max_session_retries: int = 8
    max_chunk_retries: int = 5


class Pipeline:
    """High-level orchestrator: download → verify → tile."""

    def __init__(self, config: Optional[PipelineConfig] = None) -> None:
        self.config = config or PipelineConfig()
        self.downloads_dir = self.config.workspace / self.config.downloads_subdir
        self.tiles_root = self.config.workspace / self.config.tiles_subdir
        self.downloads_dir.mkdir(parents=True, exist_ok=True)
        self.tiles_root.mkdir(parents=True, exist_ok=True)
        self._mast: Optional[MastClient] = None
        log.info(f"Pipeline workspace: {self.config.workspace}")

    # ------------------------------------------------------------------
    # Public methods
    # ------------------------------------------------------------------
    def run_url(self, url: str, filename: Optional[str] = None,
                expected_sha256: Optional[str] = None) -> dict:
        """Download + verify + tile a single direct URL."""
        filename = filename or url.rsplit("/", 1)[-1].split("?")[0] or "download.bin"
        dest = self.downloads_dir / filename
        tiles_dir = self.tiles_root / Path(filename).stem

        downloader = ResumableDownloader(
            url=url, dest_path=dest,
            chunk_size=self.config.chunk_size,
            max_session_retries=self.config.max_session_retries,
            max_chunk_retries=self.config.max_chunk_retries,
        )
        downloader.download()

        ok = FileVerifier.verify(dest, expected_size=downloader.remote_size,
                                 expected_sha256=expected_sha256)
        if not ok:
            raise IOError(f"Verification failed for {dest}")

        tiler = MemoryMappedTiler(
            tile_size=self.config.tile_size,
            overlap=self.config.tile_overlap,
            skip_empty=self.config.skip_empty_tiles,
        )
        manifest = tiler.tile(dest, tiles_dir)
        return {
            "source_url": url,
            "downloaded_file": str(dest),
            "tiles_dir": str(tiles_dir),
            "manifest": str(manifest),
            "verified": True,
        }

    def run_mast(self, target_name: str, instrument: str = "NIRCAM",
                 filter_name: Optional[str] = None,
                 max_observations: int = 3) -> list[dict]:
        """Search MAST, download every i2d.fits, verify, tile each."""
        if self._mast is None:
            self._mast = MastClient(api_token=self.config.mast_api_token)

        observations = self._mast.search_by_target(
            target_name, instrument=instrument, filter_name=filter_name,
            limit=max_observations,
        )
        if not observations:
            log.warning("No observations found.")
            return []

        results: list[dict] = []
        for obs in observations:
            try:
                products = self._mast.list_i2d_products(obs)
            except Exception as exc:
                log.error(f"Failed to list products for obs {obs.get('obsid')}: {exc}")
                continue
            for product in products:
                url = self._mast.direct_url(product)
                filename = product.get("productFilename") or url.rsplit("/", 1)[-1].split("?")[0]
                # MAST productFilename sometimes contains path separators.
                filename = Path(filename).name
                try:
                    result = self.run_url(url, filename=filename)
                    results.append(result)
                except Exception as exc:
                    log.error(f"Failed to process {filename}: {exc}")
        return results

log.info("Pipeline class loaded.")

## Step 10 — Configuration

Edit the cell below to point the pipeline at your workspace, set tile
parameters, and (optionally) provide a MAST API token for proprietary data.

In [ ]:
# ==================== USER CONFIGURATION ====================

# Where to store everything. Defaults to your Drive if mounted, else /content/space_data.
WORKSPACE = DEFAULT_WORKSPACE

# Tile parameters
TILE_SIZE = 1024           # px — 1024 is a good balance for ML workflows
TILE_OVERLAP = 64          # px — useful for stitching / overlap-aware inference
SKIP_EMPTY_TILES = False   # True = drop tiles whose pixel sum is 0 (saves disk)

# Downloader tuning (defaults are good for Colab's network)
CHUNK_SIZE = 8 * 1024 * 1024   # 8 MiB
MAX_SESSION_RETRIES = 8
MAX_CHUNK_RETRIES = 5

# MAST API token (optional — only needed for proprietary JWST data)
# Get one at: https://auth.mast.stsci.edu/
MAST_API_TOKEN = None  # or paste your token string here

# ==================== BUILD PIPELINE ====================
config = PipelineConfig(
    workspace=WORKSPACE,
    tile_size=TILE_SIZE,
    tile_overlap=TILE_OVERLAP,
    skip_empty_tiles=SKIP_EMPTY_TILES,
    mast_api_token=MAST_API_TOKEN,
    chunk_size=CHUNK_SIZE,
    max_session_retries=MAX_SESSION_RETRIES,
    max_chunk_retries=MAX_CHUNK_RETRIES,
)
pipeline = Pipeline(config=config)
log.success(f"Pipeline ready. Workspace: {WORKSPACE}")

## Step 11 — Run: Direct URL Download

Use this method when you already know the exact URL of the file you want.
The example below uses the **correct** MAST direct-download endpoint
(`/api/v0.1/Download/file?uri=...`).

> 💡 The notebook you started from had the wrong endpoint (`/public/file_lookup`).
> The correct one is `https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/<filename>`.

In [ ]:
# Example: a JWST NIRCam F200W calibrated mosaic from MAST.
# This is the CORRECT direct-download URL format.
DIRECT_URL = "https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/jw02731-o001_t017_nircam_clear-f187n_i2d.fits"
OUTPUT_FILENAME = "jwst_ngc3324_f187n_i2d.fits"

# Optional: paste a known SHA-256 to verify against (leave None to skip).
EXPECTED_SHA256 = None

result = pipeline.run_url(
    url=DIRECT_URL,
    filename=OUTPUT_FILENAME,
    expected_sha256=EXPECTED_SHA256,
)
print(json.dumps(result, indent=2))

## Step 12 — Run: MAST Search + Bulk Download

Use this method when you want to find files by target name. The pipeline
will:

1. Query MAST for matching observations.
2. For each observation, list its Level-3 `i2d.fits` products.
3. Download each product via the resumable downloader (with full retry
   support — astroquery is only used for *discovery*, not for the bytes).
4. Verify and tile each file.

This is the **recommended** method because it's resilient to MAST URL
changes — only the `astroquery` library needs to keep up with the API,
and your code stays stable.

In [ ]:
# Search for the Carina Nebula (NGC 3324) in NIRCam F187N filter.
# Change these to match your science goal.
TARGET_NAME = "NGC 3324"
INSTRUMENT = "NIRCAM"
FILTER_NAME = "F187N"     # Set to None to disable the filter criterion
MAX_OBSERVATIONS = 2      # Cap to keep Colab runtime bounded

results = pipeline.run_mast(
    target_name=TARGET_NAME,
    instrument=INSTRUMENT,
    filter_name=FILTER_NAME,
    max_observations=MAX_OBSERVATIONS,
)
print(f"\nProcessed {len(results)} file(s).")
for r in results:
    print(json.dumps(r, indent=2))

## Step 13 — Inspect Results

Quick utility cells to list what was downloaded and preview a tile.

In [ ]:
# List downloaded source files
print("=== Downloads ===")
for p in sorted(pipeline.downloads_dir.glob("*")):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:60s}  {size_mb:10.1f} MB")

print("\n=== Tile directories ===")
for d in sorted(pipeline.tiles_root.glob("*")):
    if d.is_dir():
        tiles = list(d.glob("tile_*.tif"))
        manifest = d / "manifest.csv"
        print(f"  {d.name:40s}  tiles={len(tiles):5d}  manifest={'yes' if manifest.exists() else 'no'}")

In [ ]:
# Preview a single tile (reads only that tile from disk — safe for huge datasets)
import matplotlib.pyplot as plt
import tifffile

# Pick the first tile from the first tile directory.
tile_dirs = [d for d in pipeline.tiles_root.glob("*") if d.is_dir()]
if tile_dirs:
    sample_tile = sorted(tile_dirs[0].glob("tile_*.tif"))[0]
    print(f"Previewing: {sample_tile}")
    arr = tifffile.imread(str(sample_tile))
    print(f"Shape: {arr.shape}, dtype: {arr.dtype}")

    # For scientific data, percentile-stretch for display.
    display = arr.astype(np.float32)
    if display.size > 0:
        lo, hi = np.percentile(display[display > 0], [1, 99]) if np.any(display > 0) else (0, 1)
        display = np.clip((display - lo) / max(hi - lo, 1e-9), 0, 1)

    plt.figure(figsize=(6, 6))
    plt.imshow(display, cmap="gray", origin="lower")
    plt.title(sample_tile.name)
    plt.axis("off")
    plt.colorbar(shrink=0.7)
    plt.show()
else:
    print("No tile directories found yet — run Step 11 or 12 first.")

## Troubleshooting

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| `Range requests rejected` warning | Server doesn't support resume | Pipeline auto-recovers by restarting from 0; just let it run. |
| `SHA-256 mismatch` after download | Network corruption or proxy interference | Delete the `.part` file and re-run; the download starts fresh. |
| `Readability check failed` for FITS | File is truncated or wrong format | Verify the URL points to a real `i2d.fits` (Level 3) product, not raw `uncal`. |
| `Readability check failed` for TIFF | Multi-page or BigTIFF variant the reader can't mmap | Open manually with `tifffile.TiffFile(path)` to inspect; some JPEG-in-TIFFs need a different reader. |
| MAST returns 0 observations | Target name spelled differently in archive | Try `target_name = "CARINA NEBULA"` or query by `obs_id` instead. |
| `401 Unauthorized` from MAST | Data is proprietary | Set `MAST_API_TOKEN` in Step 10 to your MAST token. |
| Colab kernel dies during tiling | `tifffile.asarray(out="memmap")` fell back to in-memory | Use a smaller `TILE_SIZE` or upgrade `tifffile`. |
| Drive mount hangs | Browser auth pop-up blocked | Re-run Step 3 — Colab will retry the auth flow. |

---

### Output layout

```
<workspace>/
├── downloads/                    ← verified source files (FITS / TIFF)
│   ├── jwst_ngc3324_f187n_i2d.fits
│   └── ...
└── tiles/
    └── jwst_ngc3324_f187n_i2d/
        ├── manifest.csv          ← tile_index, y, x, height, width, dtype, source, tile_file
        ├── tile_00000_y000000_x000000.tif
        ├── tile_00001_y000000_x001024.tif
        └── ...
```

The `manifest.csv` is the single source of truth for downstream code — read
it first, then load only the tiles you need.